# StatsBomb Data Exploration

**Goal**: Understand what penalty data we can access before writing the extractor.

In [1]:
from statsbombpy import sb
import pandas as pd
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('Setup complete')

Setup complete


## 1. What competitions are available?

In [2]:
comps = sb.competitions()
print(f'Total competition-seasons: {len(comps)}')
comps[['competition_id', 'season_id', 'competition_name', 'season_name']]

Total competition-seasons: 75


,competition_id,season_id,competition_name,season_name
0,9,281,1. Bundesliga,2023/2024
1,9,27,1. Bundesliga,2015/2016
2,1267,107,African Cup of Nations,2023
3,16,4,Champions League,2018/2019
4,16,1,Champions League,2017/2018
...,...,...,...,...
70,35,75,UEFA Europa League,1988/1989
71,53,315,UEFA Women's Euro,2025
72,53,106,UEFA Women's Euro,2022
73,72,107,Women's World Cup,2023


## 2. Filter to penalty-rich competitions

In [3]:
target_comps = comps[
    comps['competition_name'].str.contains(
        'World Cup|UEFA Euro|Champions League|Copa America',
        case=False,
        na=False
    )
]
target_comps[['competition_id', 'season_id', 'competition_name', 'season_name']]

,competition_id,season_id,competition_name,season_name
3,16,4,Champions League,2018/2019
4,16,1,Champions League,2017/2018
5,16,2,Champions League,2016/2017
6,16,27,Champions League,2015/2016
7,16,26,Champions League,2014/2015
8,16,25,Champions League,2013/2014
9,16,24,Champions League,2012/2013
10,16,23,Champions League,2011/2012
11,16,22,Champions League,2010/2011
12,16,21,Champions League,2009/2010


## 3. Look at the 2022 World Cup matches

World Cup 2022 → `competition_id=43`, `season_id=106`

In [4]:
matches = sb.matches(competition_id=43, season_id=106)
print(f'Total matches: {len(matches)}')
matches[['match_id', 'match_date', 'home_team', 'away_team', 'home_score', 'away_score']].head(10)

Total matches: 64


,match_id,match_date,home_team,away_team,home_score,away_score
0,3857256,2022-12-02,Serbia,Switzerland,2,3
1,3869151,2022-12-03,Argentina,Australia,2,1
2,3857257,2022-11-30,Australia,Denmark,1,0
3,3857258,2022-11-24,Brazil,Serbia,2,0
4,3857288,2022-11-26,Tunisia,Australia,0,1
5,3857267,2022-11-29,Ecuador,Senegal,1,2
6,3869321,2022-12-09,Netherlands,Argentina,2,2
7,3857287,2022-11-24,Uruguay,South Korea,0,0
8,3869486,2022-12-10,Morocco,Portugal,1,0
9,3869685,2022-12-18,Argentina,France,3,3


## 4. Find the 2022 final (Argentina vs France)

This match had a famous penalty shootout — perfect for exploring.

In [5]:
final = matches[
    (matches['home_team'].isin(['Argentina', 'France'])) &
    (matches['away_team'].isin(['Argentina', 'France']))
]
final[['match_id', 'home_team', 'away_team', 'home_score', 'away_score']]

,match_id,home_team,away_team,home_score,away_score
9,3869685,Argentina,France,3,3


In [6]:
final_match_id = final.iloc[0]['match_id']
events = sb.events(match_id=final_match_id)
print(f'Total events in the final: {len(events)}')
print(f'Event types: {sorted(events["type"].unique())}')

Total events in the final: 4407
Event types: ['50/50', 'Bad Behaviour', 'Ball Receipt*', 'Ball Recovery', 'Block', 'Carry', 'Clearance', 'Dispossessed', 'Dribble', 'Dribbled Past', 'Duel', 'Foul Committed', 'Foul Won', 'Goal Keeper', 'Half End', 'Half Start', 'Injury Stoppage', 'Interception', 'Miscontrol', 'Offside', 'Pass', 'Player Off', 'Player On', 'Pressure', 'Shield', 'Shot', 'Starting XI', 'Substitution', 'Tactical Shift']


## 5. Find all penalty shots in this match

In [9]:
shots = events[events['type'] == 'Shot']
print(f'Total shots: {len(shots)}')

penalties = shots[shots['shot_type'] == 'Penalty']
print(f'Penalties: {len(penalties)}')

penalties[['minute', 'period', 'player', 'team', 'shot_outcome', 'shot_body_part']]

Total shots: 38
Penalties: 11


,minute,period,player,team,shot_outcome,shot_body_part
4210,22,1,Lionel Andrés Messi Cuccittini,Argentina,Goal,Left Foot
4219,79,2,Kylian Mbappé Lottin,France,Goal,Right Foot
4234,117,4,Kylian Mbappé Lottin,France,Goal,Right Foot
4237,120,5,Kylian Mbappé Lottin,France,Goal,Right Foot
4238,121,5,Lionel Andrés Messi Cuccittini,Argentina,Goal,Left Foot
4239,121,5,Kingsley Coman,France,Saved,Right Foot
4240,122,5,Paulo Bruno Exequiel Dybala,Argentina,Goal,Left Foot
4241,123,5,Aurélien Djani Tchouaméni,France,Off T,Right Foot
4242,124,5,Leandro Daniel Paredes,Argentina,Goal,Right Foot
4243,125,5,Randal Kolo Muani,France,Goal,Right Foot


## 6. Inspect one full penalty event

This shows every column StatsBomb provides for a penalty.

In [10]:
sample_pen = penalties.iloc[0]
print('--- All non-null fields for one penalty ---')
sample_pen.dropna()

--- All non-null fields for one penalty ---


duration                                            0.625635
id                      6d527ebc-a948-4cd8-ac82-daced35bb715
index                                                    771
location                                       [108.0, 40.0]
match_id                                             3869685
minute                                                    22
period                                                     1
play_pattern                                           Other
player                        Lionel Andrés Messi Cuccittini
player_id                                             5503.0
position                                          Right Wing
possession                                                32
possession_team                                    Argentina
possession_team_id                                       779
related_events        [c9b8e568-dcdc-4302-9683-0e9e9a55a42a]
second                                                    24
shot_body_part          

## 7. Examine the freeze frame (player positions at shot moment)

The freeze frame tells us where every player was when the shot was taken — including the keeper.

In [11]:
freeze_frame = sample_pen.get('shot_freeze_frame')
if isinstance(freeze_frame, list):
    print(f'Players in freeze frame: {len(freeze_frame)}')
    for player in freeze_frame:
        pos = player.get('position', {}).get('name')
        if pos == 'Goalkeeper':
            print(f"Keeper: {player['player']['name']} | Teammate: {player['teammate']} | Location: {player['location']}")
else:
    print('No freeze frame for this event')

No freeze frame for this event


## 8. Understand shot locations

StatsBomb pitch is 120×80. The goal is at x=120, y between 36 and 44, z up to ~2.67m.

`location` → where shot was taken from (penalty spot ≈ [108, 40])

`shot_end_location` → where ball ended ([x, y, z] — z is height)

In [12]:
for i, pen in penalties.iterrows():
    print(f"{pen['player']:30} | Start: {pen['location']} | End: {pen.get('shot_end_location')} | Outcome: {pen['shot_outcome']}")

Lionel Andrés Messi Cuccittini | Start: [108.0, 40.0] | End: [120.0, 41.8, 0.2] | Outcome: Goal
Kylian Mbappé Lottin           | Start: [108.0, 40.0] | End: [120.0, 37.3, 0.3] | Outcome: Goal
Kylian Mbappé Lottin           | Start: [108.0, 40.0] | End: [120.0, 36.7, 1.1] | Outcome: Goal
Kylian Mbappé Lottin           | Start: [108.1, 40.1] | End: [120.0, 37.6, 1.3] | Outcome: Goal
Lionel Andrés Messi Cuccittini | Start: [108.1, 40.1] | End: [120.0, 38.3, 0.2] | Outcome: Goal
Kingsley Coman                 | Start: [108.1, 40.1] | End: [118.8, 38.2, 0.9] | Outcome: Saved
Paulo Bruno Exequiel Dybala    | Start: [108.1, 40.1] | End: [120.0, 40.4, 0.2] | Outcome: Goal
Aurélien Djani Tchouaméni      | Start: [108.1, 40.1] | End: [120.0, 35.6, 0.2] | Outcome: Off T
Leandro Daniel Paredes         | Start: [108.1, 40.1] | End: [120.0, 37.6, 0.2] | Outcome: Goal
Randal Kolo Muani              | Start: [108.1, 40.1] | End: [120.0, 39.3, 1.6] | Outcome: Goal
Gonzalo Ariel Montiel          | Start

## 9. Quick sanity check — penalties across one full competition

In [ ]:
# Count penalties across the whole WC 2022 (this will take ~2-3 minutes)
from tqdm.notebook import tqdm

pen_count = 0
shootout_count = 0

for match_id in tqdm(matches['match_id'].head(20), desc='First 20 matches'):
    try:
        ev = sb.events(match_id=match_id)
        if 'shot_type' in ev.columns:
            pens = ev[(ev['type'] == 'Shot') & (ev['shot_type'] == 'Penalty')]
            pen_count += len(pens)
            shootout_count += (pens['period'] == 5).sum()
    except Exception as e:
        print(f'Match {match_id} failed: {e}')

print(f'\nTotal penalties (first 20 matches): {pen_count}')
print(f'  Shootout penalties: {shootout_count}')
print(f'  In-game penalties:  {pen_count - shootout_count}')

## 10. Check if Asian and African comps are present

In [ ]:
comps = sb.competitions()

# Search broadly for African and Asian tournaments
keywords = ['Africa', 'AFC', 'Asian', 'Asia', 'CAF', 'Nations']

for kw in keywords:
    matches = comps[comps['competition_name'].str.contains(kw, case=False, na=False)]
    if len(matches) > 0:
        print(f"\n--- Matches for '{kw}' ---")
        print(matches[['competition_id', 'season_id', 'competition_name', 'season_name']].to_string(index=False))
    else:
        print(f"No matches for '{kw}'")


--- Matches for 'Africa' ---
 competition_id  season_id       competition_name season_name
           1267        107 African Cup of Nations        2023
No matches for 'AFC'
No matches for 'Asian'
No matches for 'Asia'
No matches for 'CAF'

--- Matches for 'Nations' ---
 competition_id  season_id       competition_name season_name
           1267        107 African Cup of Nations        2023


## 11. Quality check post extraction of data

In [21]:
import pandas as pd

df = pd.read_csv("C:\\Users\\NABARUN\\Code\\penalty-simulator\\data\\raw\\penalties_raw.csv")

# Check the most important columns aren't full of nulls
print("--- Null counts in key columns ---")
print(df[['taker_name', 'keeper_name', 'shot_outcome','end_x', 'end_y', 'end_z', 'shot_xg']].isna().sum())
print("\n--- Shot outcome distribution ---")
print(df['shot_outcome'].value_counts())
print("\n--- End location ranges ---")
print(df[['end_x', 'end_y', 'end_z']].describe())

print("\n--- Sample of 5 random penalties ---")
print(df.sample(5)[['taker_name', 'keeper_name', 'shot_outcome', 'end_y', 'end_z', 'is_shootout']].to_string(index=False))

--- Null counts in key columns ---
taker_name      0
keeper_name     0
shot_outcome    0
end_x           0
end_y           0
end_z           2
shot_xg         0
dtype: int64

--- Shot outcome distribution ---
shot_outcome
Goal             866
Saved            204
Off T             46
Post              42
Saved to Post      8
Wayward            2
Name: count, dtype: int64

--- End location ranges ---
             end_x        end_y        end_z
count  1168.000000  1168.000000  1166.000000
mean    119.836130    39.811815     0.949314
std       0.438795     2.810117     0.907118
min     111.400000    34.600000     0.000000
25%     120.000000    37.100000     0.200000
50%     120.000000    39.200000     0.700000
75%     120.000000    42.800000     1.400000
max     120.000000    45.500000     6.500000

--- Sample of 5 random penalties ---
                 taker_name                keeper_name shot_outcome  end_y  end_z  is_shootout
          Gareth Frank Bale                  Jan Oblak     

## Done!

**Things to confirm:**
- You see competitions listed
- The 2022 final has 9+ penalty events (1 in-game from Argentina + 1 from France + shootout penalties)
- Freeze frames contain goalkeeper info
- `shot_end_location` has 3 values (x, y, z)